# 📘 YOLOv8 License Plate Detection - Colab Ready

In [1]:

# STEP 1: Install ultralytics and check GPU
!pip install ultralytics --quiet
from IPython.display import clear_output
clear_output()

import torch
print("✅ Installed ultralytics.")
print("GPU available:", torch.cuda.is_available())


✅ Installed ultralytics.
GPU available: False


In [2]:

# STEP 2: Mount Google Drive and unzip dataset
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/Colab Notebooks/dataset.zip" -d /content/
print("✅ Dataset unzipped.")


Mounted at /content/drive
✅ Dataset unzipped.


In [ ]:

# STEP 3: Create data.yaml
yaml_content = '''path: /content/dataset
train: images/train
val: images/val
nc: 1
names: ['license_plate']'''
with open("data.yaml", "w") as f:
    f.write(yaml_content)
print("✅ data.yaml created.")


In [ ]:

# STEP 4: Train YOLOv8
from ultralytics import YOLO
model = YOLO("yolov8n.pt")

model.train(
    data="data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    project="trained-models",
    name="license-plate",
    workers=2
)


In [ ]:

# STEP 5: Create test images
import os
from PIL import Image, ImageDraw

os.makedirs("/content/test", exist_ok=True)

def create_plate(text, filename):
    img = Image.new("RGB", (220, 100), "white")
    draw = ImageDraw.Draw(img)
    draw.rectangle([30, 30, 190, 70], outline="black", width=2)
    draw.text((60, 40), text, fill="black")
    img.save("/content/test/" + filename)

create_plate("51F-1001", "test1.jpg")
create_plate("29A-5555", "test2.jpg")
create_plate("63C-8888", "test3.jpg")
print("✅ Test images created.")


In [ ]:

# STEP 6: Run prediction on test images
model = YOLO("/content/trained-models/license-plate/weights/best.pt")

for img in os.listdir("/content/test"):
    if img.endswith(".jpg"):
        path = "/content/test/" + img
        print("🔍 Predicting:", img)
        model.predict(path, save=True, conf=0.4)
print("✅ Predictions saved to /content/runs/detect/predict/")
